In [9]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

FILES = {
    'Sarima': 'sarima_predictions.csv',
    'gbt': 'gbt_predictions15days.csv',
    'lstm': 'model_predictions-lstm-aws.csv',
    'gru': 'model_predictions-gru-aws.csv',
    'adaboost': 'model_predictions_stacked_adaboost.csv',
    'random_forest': 'model_predictions_stacked_rf.csv',
    'stacking_gbt': 'model_predictions_stacked_gbt.csv'
}

ACTUAL_CANDIDATES = ['actual', 'actual_label', 'y_true', 'label']
PRED_CANDIDATES = ['predicted', 'prediction', 'pred','predicted_label', 'y_pred','stacked_prediction']

DATE_CANDIDATES = ['date', 'Date', 'prediction_date', 'prediction_date_lstm', 'prediction_date_gru']

# Filter threshold
FILTER_DATE = pd.to_datetime('2025-05-15')

os.makedirs('confusion_plots', exist_ok=True)

for name, fname in FILES.items():
    if not os.path.exists(fname):
        print(f"Skipping {name}: file {fname} not found")
        continue

    df = pd.read_csv(fname)

    # Find date column
    date_col = next((c for c in DATE_CANDIDATES if c in df.columns), None)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col] > FILTER_DATE]
    else:
        print(f"Warning: {name} has no recognized date column; using all rows")

    if df.empty:
        print(f"Skipping {name}: no rows after {FILTER_DATE.date()}")
        continue

    # find actual and predicted columns
    actual_col = next((c for c in ACTUAL_CANDIDATES if c in df.columns), None)
    pred_col = next((c for c in PRED_CANDIDATES if c in df.columns), None)

    # if predicted probability exists but not hard prediction, try thresholding
    if pred_col is None:
        prob_cols = [c for c in df.columns if 'prob' in c.lower() or 'proba' in c.lower()]
        if prob_cols:
            pred_col = 'derived_pred'
            df[pred_col] = (df[prob_cols[0]] >= 0.5).astype(int)

    if actual_col is None or pred_col is None:
        print(f"Skipping {name}: couldn't find actual/predicted columns. cols={list(df.columns)}")
        continue

    y_true = df[actual_col].astype(int)
    y_pred = df[pred_col].astype(int)

    labels = np.unique(np.concatenate([y_true.unique(), y_pred.unique()]))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix: {name}')
    out_path = os.path.join('confusion_plots', f'confusion_{name}.png')
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

    report = classification_report(y_true, y_pred, zero_division=0)
    txt_path = os.path.join('confusion_plots', f'classification_report_{name}.txt')
    with open(txt_path, 'w') as f:
        f.write(report)

    print(f"Saved {out_path} and {txt_path}")

print('Done')


Saved confusion_plots\confusion_Sarima.png and confusion_plots\classification_report_Sarima.txt
Saved confusion_plots\confusion_gbt.png and confusion_plots\classification_report_gbt.txt
Saved confusion_plots\confusion_lstm.png and confusion_plots\classification_report_lstm.txt
Saved confusion_plots\confusion_gru.png and confusion_plots\classification_report_gru.txt
Saved confusion_plots\confusion_adaboost.png and confusion_plots\classification_report_adaboost.txt
Saved confusion_plots\confusion_random_forest.png and confusion_plots\classification_report_random_forest.txt
Saved confusion_plots\confusion_stacking_gbt.png and confusion_plots\classification_report_stacking_gbt.txt
Done


In [ ]:
import os
import pandas as pd

FILES = [
    'gbt_predictions15days.csv',
    'model_predictions-lstm-aws.csv',
    'model_predictions-gru-aws.csv',
    'model_predictions_stacked_adaboost.csv',
    'model_predictions_stacked_rf.csv',
    'model_predictions_stacked_gbt.csv'
]

DATE_CANDIDATES = ['date', 'Date', 'prediction_date', 'prediction_date_lstm', 'prediction_date_gru']
FILTER_DATE = pd.to_datetime('2025-05-14')

for fname in FILES:
    if not os.path.exists(fname):
        print(f"Skipping {fname}: file not found")
        continue

    df = pd.read_csv(fname)

    # find date column
    date_col = next((c for c in DATE_CANDIDATES if c in df.columns), None)
    if date_col is None:
        print(f"Skipping {fname}: no date column found")
        continue

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df[df[date_col] > FILTER_DATE]

    if df.empty:
        print(f"Warning: {fname} has no rows after {FILTER_DATE.date()}")
        continue

    # Overwrite the original CSV
    df.to_csv(fname, index=False)
    print(f"Filtered {fname} to only include rows after {FILTER_DATE.date()}")


Filtered gbt_predictions15days.csv to only include rows after 2025-05-15
Filtered model_predictions-lstm-aws.csv to only include rows after 2025-05-15
Filtered model_predictions-gru-aws.csv to only include rows after 2025-05-15
Filtered model_predictions_stacked_adaboost.csv to only include rows after 2025-05-15
Filtered model_predictions_stacked_rf.csv to only include rows after 2025-05-15
Filtered model_predictions_stacked_gbt.csv to only include rows after 2025-05-15
